# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary. 

- **Problem type:** Supervised **regression**—predict continuous listing **price** (`price`) from predictors in `vehicles.csv`.
- **Inputs:** Vehicle and listing **features** (e.g., `year`, `odometer`, `manufacturer`, `model`, `condition`, `fuel`, `transmission`, `drive`, `type`).
- **Process:** **EDA** → **clean** missing/invalid values and outliers → **encode** categoricals for `sklearn` → fit regression models with **train/test split** and **cross-validation**.
- **Evaluation:** Compare models using **R²**, **RMSE**, and/or **MAE** on held-out data.
- **Interpretation:** Use **coefficients** or **feature importance** to rank price drivers and inform dealership **inventory and pricing** decisions.

In [ ]:
# Run this notebook from the code_files/ directory
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv("data/vehicles.csv")
df.head()

### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

**Planned steps (CRISP-DM — Data Understanding)**

- **Collect / load data:** Read `data/vehicles.csv`, confirm row count and that columns match expectations for pricing analysis.
- **Describe data:** Review `dtypes`, unique values, and summary statistics for numeric fields (`price`, `year`, `odometer`) and categorical fields (`manufacturer`, `condition`, `fuel`, etc.).
- **Explore data:** Visualize distributions and relationships to `price` (histograms, boxplots, correlation for numeric features, group means for categories).
- **Verify data quality:** Quantify missing values, invalid targets (e.g., $0 or extreme prices), implausible `year`/`odometer`, duplicate `id`s, and high-cardinality fields (`model`, `VIN`) that may need grouping or exclusion from modeling.
- **Connect to business goals:** Note which complete, interpretable features are likely price drivers vs. which need cleaning or imputation before modeling.

In [ ]:
# Structure and basic summaries (avoid dumping full info/describe output)
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
display(df.head())
print("\nColumn dtypes:")
print(df.dtypes)
print("\nNumeric summary (price, year, odometer):")
display(df[["price", "year", "odometer"]].describe().round(2))

In [ ]:
# Missing values and target integrity
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("Missing values (% of rows):")
print(missing_pct.round(1))

print("\nDuplicate listing ids:", df['id'].duplicated().sum())

print("\nPrice checks:")
print(df['price'].describe())
print("Listings with price <= 0:", (df['price'] <= 0).sum())
print("Listings with price > 1,000,000:", (df['price'] > 1_000_000).sum())

print("\nYear range:", df['year'].min(), "–", df['year'].max())
print("Odometer max:", df['odometer'].max())

In [ ]:
# Explore distributions and bivariate relationships with price
cols = ['price', 'year', 'odometer']
df_plot = df[cols].dropna()
df_plot = df_plot[(df_plot['price'] > 0) & (df_plot['price'] < 200_000)]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(df_plot["price"], bins=50, ax=axes[0])
axes[0].set_title("Listing price distribution")
axes[0].set_xlabel("Price (USD)")
sns.histplot(df_plot["year"], bins=30, ax=axes[1])
axes[1].set_title("Model year distribution")
axes[1].set_xlabel("Year")
sns.histplot(df_plot["odometer"], bins=50, ax=axes[2])
axes[2].set_title("Odometer distribution")
axes[2].set_xlabel("Miles")
plt.tight_layout()
plt.show()

numeric = df[["price", "year", "odometer"]].apply(pd.to_numeric, errors="coerce")
print("Correlation with price (pairwise complete rows):")
print(numeric.corr()["price"].sort_values(ascending=False).round(3))

**Initial findings and implications**

- **Dataset size:** ~427K listings with 18 columns; `id` and `state`/`region` are complete, while **`size` (~72% missing)**, **`VIN` (~38%)**, **`condition` / `cylinders` (~41%)**, and **`drive` / `paint_color` (~31%)** will need imputation, a missing indicator, or exclusion depending on the model.
- **Target (`price`):** Always present but **not always valid**—many **$0** listings and a few **extreme high values** look like placeholders or errors; modeling should filter or cap outliers and drop non-positive prices.
- **Core predictors:** **`year`** and **`odometer`** are largely complete and align with the business question (newer, lower-mileage vehicles tend to command higher prices); expect them to be strong drivers after cleaning.
- **Categorical signal:** `manufacturer`, `model`, `fuel`, `transmission`, `type`, and `title_status` are mostly populated and suitable for one-hot or target encoding after cardinality control (`model` is very high-cardinality).
- **Geography:** `state` and `region` support optional segmentation but may add noise nationally; useful for local inventory decisions.
- **Next phase (Data Preparation):** Define cleaning rules (price/year/odometer bounds), handle missingness, encode categoricals, and optionally engineer **vehicle age** from `year` before `sklearn` pipelines.

In [ ]:
# Categorical variables vs. price (boxplots for continuous target, barplot for median by group)
eda = df[(df["price"] > 5_000) & (df["price"] < 200_000)].copy()
cat_cols = ["fuel", "condition", "transmission", "type"]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.flat, cat_cols):
    top = eda[col].dropna().value_counts().head(6).index
    subset = eda[eda[col].isin(top)]
    sns.boxplot(data=subset, x=col, y="price", order=top, ax=ax)
    ax.set_title(f"Price by {col} (top 6 categories)")
    ax.set_xlabel(col.title())
    ax.set_ylabel("Price (USD)")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
top_makes = eda["manufacturer"].value_counts().head(8).index
make_med = (
    eda[eda["manufacturer"].isin(top_makes)]
    .groupby("manufacturer")["price"]
    .median()
    .reindex(top_makes)
)
sns.barplot(x=make_med.index, y=make_med.values, ax=ax)
ax.set_title("Median listing price by manufacturer (top 8)")
ax.set_xlabel("Manufacturer")
ax.set_ylabel("Median price (USD)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`. 

**Preparation plan**

- **Select / drop columns:** Remove identifiers and leaky or sparse fields (`id`, `VIN`, `region`); defer `model` until cardinality is controlled.
- **Clean rows:** Keep plausible listings (positive price in a reasonable range, sensible `year` and `odometer`).
- **Construct features:** Add `vehicle_age` from `year`; keep numeric `year` and `odometer` for modeling.
- **Impute and encode:** Median imputation + scaling for numerics; most-frequent imputation + one-hot encoding for categoricals.
- **Split data:** Hold out a test set for final evaluation; fit preprocessors on training data only (via `Pipeline` / `ColumnTransformer`).

In [ ]:
# Row-level cleaning and feature construction
CURRENT_YEAR = int(df['year'].dropna().max())

cars = df.drop(columns=['id', 'VIN', 'region'], errors='ignore').copy()

cars = cars[(cars['price'] > 5_000) & (cars['price'] < 200_000)]
cars = cars[(cars['year'] >= 1990) & (cars['year'] <= CURRENT_YEAR)]
cars = cars[cars['odometer'].isna() | ((cars['odometer'] >= 0) & (cars['odometer'] <= 500_000))]

cars['vehicle_age'] = CURRENT_YEAR - cars['year']

print(f"Rows after cleaning: {len(cars):,} (from {len(df):,})")
cars.head()

In [ ]:
# Modeling feature sets (interpretable, mostly complete columns)
numeric_features = ['year', 'odometer', 'vehicle_age']
categorical_features = [
    'manufacturer', 'condition', 'cylinders', 'fuel',
    'title_status', 'transmission', 'drive', 'type', 'paint_color', 'state',
]
# High missingness — exclude from first-pass model
dropped_sparse = ['size']

feature_cols = numeric_features + categorical_features
X = cars[feature_cols]
y = cars['price']

print("Feature matrix shape:", X.shape)
print("\nRemaining missing (%):")
print((X.isna().mean() * 100).sort_values(ascending=False).round(1))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", ohe),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Prepared feature count:", X_train_prep.shape[1])

In [ ]:
# Optional: log-transform target for skewed price (use y_train_log in some models)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# Ready for Modeling: attach preprocessor to estimators with Pipeline([('prep', preprocessor), ('model', ...)])
modeling_data = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'preprocessor': preprocessor,
}
list(modeling_data.keys())

**Prepared dataset summary**

- **Cleaning:** Dropped invalid/low prices and extreme values; restricted `year` (1990–latest in data) and capped implausible `odometer` readings; removed ID-like columns not useful for generalization.
- **Features:** `vehicle_age` complements `year`; categoricals capture condition, powertrain, body type, title, and geography (`state`).
- **Transformations:** Numeric features are median-imputed and **standardized**; categoricals are mode-imputed and **one-hot encoded** (`handle_unknown='ignore'` for unseen categories at scoring time).
- **Modeling setup:** 80/20 **train/test** split with `random_state=42`; `preprocessor` should be **fit only on training data** inside each model `Pipeline` to avoid leakage. Optional **`log1p(price)`** target helps with right-skewed prices.
- **Follow-ups:** Add grouped `model` features, tune price/odometer cutoffs, or try `min_frequency` on one-hot encoding if dimensionality is too large.

### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

**Modeling approach**

- Wrap **`preprocessor` + estimator** in a single `Pipeline` so encoding/imputation is refit inside each CV fold.
- Compare several **regressors** (linear, regularized linear, tree ensemble) on the same train split.
- Use **5-fold cross-validation** on the training set for model selection; reserve the **test set** for final evaluation only.
- Track **R²**, **RMSE**, and **MAE** (see Evaluation for why these metrics were chosen).

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline

cv_args = dict(cv=5, scoring=['r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error'], n_jobs=-1)

candidates = {
    'linear_regression': Pipeline([
        ('prep', preprocessor),
        ('model', LinearRegression()),
    ]),
    'ridge': Pipeline([
        ('prep', preprocessor),
        ('model', Ridge()),
    ]),
    'random_forest': Pipeline([
        ('prep', preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=100,
            max_depth=20,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        )),
    ]),
}

cv_results = []
for name, pipe in candidates.items():
    scores = cross_validate(pipe, X_train, y_train, **cv_args)
    cv_results.append({
        'model': name,
        'cv_r2_mean': scores['test_r2'].mean(),
        'cv_r2_std': scores['test_r2'].std(),
        'cv_rmse_mean': -scores['test_neg_root_mean_squared_error'].mean(),
        'cv_mae_mean': -scores['test_neg_mean_absolute_error'].mean(),
    })

cv_df = pd.DataFrame(cv_results).sort_values('cv_r2_mean', ascending=False)
cv_df

In [ ]:
# Hyperparameter search on promising models (still using train data only)
ridge_search = GridSearchCV(
    Pipeline([('prep', preprocessor), ('model', Ridge())]),
    param_grid={'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    cv=5,
    scoring='r2',
    n_jobs=-1,
)
ridge_search.fit(X_train, y_train)
print('Best Ridge alpha:', ridge_search.best_params_['model__alpha'])
print('Best Ridge CV R²:', round(ridge_search.best_score_, 4))

rf_search = GridSearchCV(
    Pipeline([('prep', preprocessor), ('model', RandomForestRegressor(random_state=42, n_jobs=-1))]),
    param_grid={
        'model__n_estimators': [100, 200],
        'model__max_depth': [15, 25, None],
        'model__min_samples_leaf': [2, 5],
    },
    cv=3,
    scoring='r2',
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)
print('Best RF params:', rf_search.best_params_)
print('Best RF CV R²:', round(rf_search.best_score_, 4))

In [ ]:
# Select champion by CV R² and refit on full training set
tuned_models = {
    'ridge_tuned': ridge_search.best_estimator_,
    'random_forest_tuned': rf_search.best_estimator_,
}

best_name = cv_df.iloc[0]['model']
if cv_df.loc[cv_df['model'] == 'ridge', 'cv_r2_mean'].iloc[0] >= cv_df.loc[cv_df['model'] == 'random_forest', 'cv_r2_mean'].iloc[0]:
    champion = ridge_search.best_estimator_
    champion_name = 'ridge_tuned'
else:
    champion = rf_search.best_estimator_
    champion_name = 'random_forest_tuned'

champion.fit(X_train, y_train)
y_pred_train = champion.predict(X_train)
y_pred_test = champion.predict(X_test)

def regression_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
    }

final_metrics = pd.DataFrame({
    'train': regression_metrics(y_train, y_pred_train),
    'test': regression_metrics(y_test, y_pred_test),
}).T
print('Champion model:', champion_name)
final_metrics

### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

**Evaluation metrics (what we report and why)**

- **R² (coefficient of determination):** Share of price variation explained on the test set—useful for comparing models on the same data split.
- **RMSE (root mean squared error):** Typical error in **dollars**, penalizing large mistakes more than MAE—good when costly mispricing on expensive vehicles matters.
- **MAE (mean absolute error):** Average prediction error in **dollars**—the easiest metric to explain to dealers (“off by about $X per listing on average”).

We rely on **cross-validated R²** for model and hyperparameter selection, then report **test-set R², RMSE, and MAE** once on held-out listings to approximate real-world performance.

In [ ]:
# Hold-out comparison across tuned models
compare_rows = []
for label, est in tuned_models.items():
    est.fit(X_train, y_train)
    preds = est.predict(X_test)
    m = regression_metrics(y_test, preds)
    m['model'] = label
    compare_rows.append(m)

test_compare = pd.DataFrame(compare_rows).set_index('model')
test_compare

In [ ]:
# Diagnostic: predicted vs actual and residuals on test set
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_test, y_pred_test, alpha=0.15, s=8)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=1)
axes[0].set_xlabel('Actual price')
axes[0].set_ylabel('Predicted price')
axes[0].set_title(f'Test set — {champion_name}')

residuals = y_test - y_pred_test
sns.histplot(residuals, bins=50, ax=axes[1])
axes[1].set_xlabel("Residual (USD)")
axes[1].set_title("Residual distribution (test)")
plt.tight_layout()
plt.show()

In [ ]:
# Interpretable drivers: Ridge coefficients (magnitude) on prepared features
ridge_model = ridge_search.best_estimator_
feature_names = ridge_model.named_steps['prep'].get_feature_names_out()
coefs = ridge_model.named_steps['model'].coef_

coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
print('Top positive associations with price (Ridge):')
display(coef_df.sort_values('coefficient', ascending=False).head(10))
print('Top negative associations with price (Ridge):')
display(coef_df.sort_values('coefficient').head(10))

# Simple EDA backup: numeric partial relationships on cleaned data
for col in ['year', 'odometer', 'vehicle_age']:
    print(f"Median price by {col} quartile:")
    print(cars.groupby(pd.qcut(cars[col], 4, duplicates='drop'))['price'].median())
    print()

**Evaluation vs. business goals**

- **Model quality:** A strong model shows high **test R²** and **low RMSE/MAE** relative to alternatives, with train and test metrics that are not wildly different (large gaps suggest overfitting).
- **Useful for the dealership?** Yes, for **ranking** what tends to increase or decrease listing price and for **ballpark pricing**, especially when combined with domain judgment.
- **Key drivers (consistent with EDA + Ridge):** **Newer model year / lower vehicle age** and **lower odometer** generally align with higher prices; categorical signals (e.g., **fuel**, **drive**, **type**, **manufacturer**, **title_status**) shift price after controlling for age and mileage.
- **Limitations:** Craigslist listings are noisy (missing fields, typos, $0 placeholders removed); no trim-level options; **state/region** effects may not match a single dealership’s market; rare luxury listings can still distort errors.
- **Revisit earlier CRISP phases?** Consider adding **`model`** (grouped), richer **outlier rules**, and **local inventory data** in a future iteration; current artifacts are sufficient for a first client readout.

### Findings

Plain-language takeaways for the dealership (supported by EDA and regression models above):

1. **Buy for age and miles:** Shoppers pay more for **newer cars** and **lower odometer** readings—the clearest patterns in the data.
2. **Stock the right configuration:** Body style, fuel type, and drivetrain (e.g., trucks/4WD where demand is high) move typical prices up or down.
3. **Title status matters:** Listings with clean title indicators tend to price above salvage or problem-title vehicles when those fields are reported.
4. **Brand still counts:** Some manufacturers show higher median prices in this market even after mileage and year are considered in the model.
5. **Use predictions as a guide:** Compare your ask price to the model’s estimate; large gaps flag underpriced buys or overpriced units that may sit on the lot.

**Action items:** Focus acquisition on recent, low-mile units; verify title and condition in person; retrain the model periodically on your own sold inventory for local accuracy.

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.

**Deliverable:** Share the **Findings** section with sales and buying staff. Operational steps below.

- **Artifact:** Save the champion `Pipeline` (preprocessing + model) with `joblib` for batch scoring of inventory spreadsheets.
- **Monitoring:** Refresh quarterly; track **test MAE in dollars** on recent sales—if error drifts, revisit cleaning rules and retrain.
- **Human in the loop:** Override predictions for accidents, reconditioning, warranties, and promos not in listing data.
- **Caveats:** Craigslist listings are self-reported; national data may not match a single lot—supplement with dealership sold-price history when possible.

In [ ]:
# Optional: persist champion pipeline for deployment / batch scoring
# import joblib
# joblib.dump(champion, 'price_model_pipeline.joblib')